# Notebook 05 — Overcomplete Environment and SAE Preparation

## Project: Metamer-SAE

### Purpose

This notebook establishes a reproducible environment for training
an Overcomplete TopK Sparse Autoencoder (SAE).

It also prepares the pooled ConvNeXt-Tiny activation matrices
extracted in Notebook 03.

### Main objectives

1. Establish a Python environment compatible with Overcomplete.
2. Install and verify the pinned Overcomplete package.
3. Inspect the actual TopK SAE implementation and API.
4. Perform forward-pass, gradient and training smoke tests.
5. Load and validate the Imagenette training and validation activations.
6. Establish the SAE preprocessing strategy.
7. Prepare reproducible DataLoaders and experiment configurations.

### Experimental data

- Training: 9,469 natural-image activation vectors.
- Validation: 3,925 natural-image activation vectors.
- Feature dimension: 384.

### Important restrictions

- Do not use Feather primary evaluation or reserve data for training.
- Do not refit PCA.
- Do not extract the activations again unless an integrity problem is identified.
- Do not begin full SAE training in this notebook.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import json
import os

# ============================================================
# METAMER-SAE PROJECT DIRECTORIES
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/metamer-sae"
)

ACTIVATION_ROOT = (
    PROJECT_ROOT / "activations/natural_images"
)

TRAIN_DIR = ACTIVATION_ROOT / "train"
VALIDATION_DIR = ACTIVATION_ROOT / "validation"

PCA_RUN_ID = "pca_20261119T065920Z"

PCA_RUN_DIR = (
    PROJECT_ROOT
    / "representations"
    / "pca"
    / "runs"
    / PCA_RUN_ID
)

OUTPUT_ROOT = PROJECT_ROOT / "outputs"

SAE_PREPARATION_DIR = (
    OUTPUT_ROOT / "sae_preparation"
)

SAE_PREPARATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# VERIFY PROJECT ROOT
# ============================================================

assert PROJECT_ROOT.exists(), (
    f"Project directory not found: {PROJECT_ROOT}"
)

print("=" * 65)
print("METAMER-SAE — PROJECT DIRECTORY VERIFICATION")
print("=" * 65)

print(f"\nProject root: {PROJECT_ROOT}")
print(f"Activation directory: {ACTIVATION_ROOT}")
print(f"PCA run: {PCA_RUN_ID}")
print(f"SAE preparation output: {SAE_PREPARATION_DIR}")

# ============================================================
# VERIFY NATURAL-IMAGE ACTIVATION FILES
# ============================================================

required_activation_files = [
    TRAIN_DIR / "pooled_activations.npy",
    TRAIN_DIR / "metadata.csv",
    VALIDATION_DIR / "pooled_activations.npy",
    VALIDATION_DIR / "metadata.csv",
]

missing = [
    str(path)
    for path in required_activation_files
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Required activation files are missing:\n"
        + "\n".join(missing)
    )

print("\nNATURAL-IMAGE ACTIVATIONS")
print("-" * 65)

for path in required_activation_files:
    print(
        f"{path.name:30s} "
        f"{path.stat().st_size / (1024**2):.2f} MB"
    )

# ============================================================
# VERIFY CORE PCA ARTIFACTS
# ============================================================

required_pca_files = [
    PCA_RUN_DIR / "pca_full_384_components.joblib",
    PCA_RUN_DIR / "pca_run_config.json",
    PCA_RUN_DIR / "pca_component_retention_metrics.csv",
    PCA_RUN_DIR / "pca_training_feature_statistics.csv",
    PCA_RUN_DIR / "pca_development_embeddings.csv",
    PCA_RUN_DIR / "pca_development_pair_distances.csv",
]

missing_pca = [
    str(path)
    for path in required_pca_files
    if not path.exists()
]

if missing_pca:
    print("\nWARNING: Some core PCA artifacts are missing.")
    for path in missing_pca:
        print(" -", path)
else:
    print("\nCore PCA artifacts: VERIFIED")

# ============================================================
# INSPECT PCA CONFIGURATION
# ============================================================

pca_config_path = PCA_RUN_DIR / "pca_run_config.json"

if pca_config_path.exists():

    with open(pca_config_path, "r") as f:
        pca_config = json.load(f)

    print("\nPCA CONFIGURATION")
    print("-" * 65)

    print(json.dumps(
        pca_config,
        indent=2
    )[:4000])

print("\n" + "=" * 65)
print("PROJECT DIRECTORY CHECK COMPLETE")
print("=" * 65)

METAMER-SAE — PROJECT DIRECTORY VERIFICATION

Project root: /content/drive/MyDrive/metamer-sae
Activation directory: /content/drive/MyDrive/metamer-sae/activations/natural_images
PCA run: pca_20261119T065920Z
SAE preparation output: /content/drive/MyDrive/metamer-sae/outputs/sae_preparation

NATURAL-IMAGE ACTIVATIONS
-----------------------------------------------------------------
pooled_activations.npy         13.87 MB
metadata.csv                   1.70 MB
pooled_activations.npy         5.75 MB
metadata.csv                   0.69 MB

Core PCA artifacts: VERIFIED

PCA CONFIGURATION
-----------------------------------------------------------------
{
  "run_id": "pca_20261119T065920Z",
  "created_utc": "2026-11-19T07:07:25.690850+00:00",
  "project": "Metamer-SAE",
  "notebook": "04_pca_analysis.ipynb",
  "observer": "TorchVision ConvNeXt-Tiny, ImageNet1K_V1",
  "observer_activation": "stage 3, global-average-pooled, 384 dimensions",
  "fit_dataset": "Imagenette-320 training split only

In [ ]:
import sys
import platform
import torch

try:
    import torchvision
    torchvision_version = torchvision.__version__
except Exception as e:
    torchvision_version = f"IMPORT ERROR: {e}"

print("=" * 65)
print("METAMER-SAE — COMPUTATIONAL ENVIRONMENT")
print("=" * 65)

print(f"\nPython version:       {sys.version}")
print(f"Platform:             {platform.platform()}")
print(f"PyTorch version:      {torch.__version__}")
print(f"TorchVision version:  {torchvision_version}")

print(f"\nCUDA available:       {torch.cuda.is_available()}")
print(f"PyTorch CUDA build:   {torch.version.cuda}")

if torch.cuda.is_available():

    print(f"GPU:                  {torch.cuda.get_device_name(0)}")

    print(
        "GPU memory:          "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

# Check Overcomplete's supported Python range.

if not ((3, 8) <= sys.version_info[:2] < (3, 13)):

    raise RuntimeError(
        "Incompatible Python version for Overcomplete 0.3.0. "
        "Select a compatible Colab runtime."
    )

print("\nPython compatibility: PASSED")
print("=" * 65)

METAMER-SAE — COMPUTATIONAL ENVIRONMENT

Python version:       3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform:             Linux-6.6.159+-x86_64-with-glibc2.35
PyTorch version:      2.11.0+cu128
TorchVision version:  0.26.0+cu128

CUDA available:       True
PyTorch CUDA build:   12.8
GPU:                  Tesla T4
GPU memory:          14.56 GB

Python compatibility: PASSED


In [ ]:
%pip install --no-deps --no-cache-dir "overcomplete==0.3.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 42.0 MB/s eta 0:00:00


In [ ]:
from overcomplete.sae import TopKSAE, train_sae

In [ ]:
import inspect
import importlib.metadata as metadata

import overcomplete
from overcomplete.sae import TopKSAE, train_sae

print("=" * 65)
print("METAMER-SAE — OVERCOMPLETE API INSPECTION")
print("=" * 65)

# Package information

print("\nPACKAGE INFORMATION")
print("-" * 65)

print(
    "Installed version:",
    metadata.version("overcomplete")
)

package_metadata = metadata.metadata("overcomplete")

print(
    "Requires Python:",
    package_metadata.get("Requires-Python")
)

print(
    "Package location:",
    overcomplete.__file__
)

# Actual callable signatures

print("\nTOPKSAE SIGNATURE")
print("-" * 65)

print(inspect.signature(TopKSAE))

print("\nTOPKSAE INITIALISATION")
print("-" * 65)

print(inspect.signature(TopKSAE.__init__))

print("\nTOPKSAE FORWARD METHOD")
print("-" * 65)

print(inspect.signature(TopKSAE.forward))

print("\nTRAIN_SAE SIGNATURE")
print("-" * 65)

print(inspect.signature(train_sae))

# Inspect implementation source so we can understand
# how the model handles centering, encoding and decoding.

print("\nTOPKSAE IMPLEMENTATION SOURCE")
print("-" * 65)

try:
    print(inspect.getsource(TopKSAE))
except Exception as e:
    print("Source inspection unavailable:", e)

print("\n" + "=" * 65)
print("OVERCOMPLETE API INSPECTION COMPLETE")
print("=" * 65)

METAMER-SAE — OVERCOMPLETE API INSPECTION

PACKAGE INFORMATION
-----------------------------------------------------------------
Installed version: 0.3.0
Requires Python: >=3.8,<3.13
Package location: /usr/local/lib/python3.12/dist-packages/overcomplete/__init__.py

TOPKSAE SIGNATURE
-----------------------------------------------------------------
(input_shape, nb_concepts, top_k=None, encoder_module=None, dictionary_params=None, device='cpu')

TOPKSAE INITIALISATION
-----------------------------------------------------------------
(self, input_shape, nb_concepts, top_k=None, encoder_module=None, dictionary_params=None, device='cpu')

TOPKSAE FORWARD METHOD
-----------------------------------------------------------------
(self, x)

TRAIN_SAE SIGNATURE
-----------------------------------------------------------------
(model, dataloader, criterion, optimizer, scheduler=None, nb_epochs=20, clip_grad=1.0, monitoring=1, device='cpu')

TOPKSAE IMPLEMENTATION SOURCE
------------------------

In [ ]:
import torch
import torch.nn.functional as F

from torch.utils.data import DataLoader, TensorDataset

from overcomplete.sae import TopKSAE, train_sae

# ============================================================
# SMOKE TEST CONFIGURATION
# ============================================================

torch.manual_seed(42)

DEVICE = (
    "cuda" if torch.cuda.is_available() else "cpu"
)

INPUT_DIM = 32
NB_CONCEPTS = 64
TOP_K = 4

N_SAMPLES = 64
BATCH_SIZE = 16

# Small synthetic activation matrix

dummy_activations = torch.randn(
    N_SAMPLES,
    INPUT_DIM,
    dtype=torch.float32
)

dummy_loader = DataLoader(
    TensorDataset(dummy_activations),
    batch_size=BATCH_SIZE,
    shuffle=True
)

# ============================================================
# INITIALISE MODEL
# ============================================================

sae = TopKSAE(
    INPUT_DIM,
    nb_concepts=NB_CONCEPTS,
    top_k=TOP_K,
    device=DEVICE
)

optimizer = torch.optim.Adam(
    sae.parameters(),
    lr=5e-4
)

# Record initial parameters to check that optimisation occurs.

initial_parameters = {
    name: parameter.detach().clone()
    for name, parameter in sae.named_parameters()
}

# ============================================================
# DEFINE RECONSTRUCTION LOSS
# ============================================================

smoke_test_results = []

def smoke_test_criterion(
    x,
    x_hat,
    pre_codes,
    codes,
    dictionary
):

    assert x_hat.shape == x.shape, (
        "Reconstruction dimension mismatch."
    )

    assert torch.isfinite(x_hat).all(), (
        "Non-finite reconstruction detected."
    )

    loss = F.mse_loss(x_hat, x)

    assert torch.isfinite(loss), (
        "Non-finite loss detected."
    )

    smoke_test_results.append({
        "input_shape": tuple(x.shape),
        "reconstruction_shape": tuple(x_hat.shape),
        "pre_codes_shape": tuple(pre_codes.shape),
        "codes_shape": tuple(codes.shape),
        "loss": float(loss.detach().cpu())
    })

    return loss

# ============================================================
# RUN TWO EPOCHS OF TRAINING
# ============================================================

print("=" * 65)
print("METAMER-SAE — TOPK SAE SMOKE TEST")
print("=" * 65)

print(f"\nDevice: {DEVICE}")
print(f"Input dimension: {INPUT_DIM}")
print(f"Dictionary size: {NB_CONCEPTS}")
print(f"TopK: {TOP_K}")

logs = train_sae(
    sae,
    dummy_loader,
    smoke_test_criterion,
    optimizer,
    nb_epochs=2,
    device=DEVICE
)

# ============================================================
# VERIFY PARAMETER UPDATES
# ============================================================

total_parameter_change = 0.0

for name, parameter in sae.named_parameters():

    total_parameter_change += (
        parameter.detach() - initial_parameters[name]
    ).abs().sum().item()

assert len(smoke_test_results) > 0, (
    "The training criterion was never executed."
)

assert total_parameter_change > 0, (
    "Model parameters did not change during training."
)

assert all(
    torch.isfinite(torch.tensor(item["loss"]))
    for item in smoke_test_results
), "Non-finite training loss detected."

print("\nSMOKE TEST RESULTS")
print("-" * 65)

print("Training batches:", len(smoke_test_results))

print(
    "First recorded batch:",
    smoke_test_results[0]
)

print(
    "Total parameter change:",
    total_parameter_change
)

print("Training log type:", type(logs).__name__)

print("\n" + "=" * 65)
print("TOPK SAE SMOKE TEST PASSED")
print("=" * 65)

METAMER-SAE — TOPK SAE SMOKE TEST

Device: cuda
Input dimension: 32
Dictionary size: 64
TopK: 4
Epoch[1/2], Loss: 1.0977, R2: -0.1272, L0: 4.0000, Dead Features: 6.2%, Time: 2.7832 seconds
Epoch[2/2], Loss: 1.0804, R2: -0.1072, L0: 4.0000, Dead Features: 6.2%, Time: 0.0113 seconds

SMOKE TEST RESULTS
-----------------------------------------------------------------
Training batches: 8
First recorded batch: {'input_shape': (16, 32), 'reconstruction_shape': (16, 32), 'pre_codes_shape': (16, 64), 'codes_shape': (16, 64), 'loss': 1.1926774978637695}
Total parameter change: 60.65322634577751
Training log type: defaultdict

TOPK SAE SMOKE TEST PASSED


### Cell 7 — Inspect the inherited SAE implementation

In [ ]:
import inspect
import torch

from overcomplete.sae import TopKSAE

# ============================================================
# INITIALISE AN ARCHITECTURE INSPECTION MODEL
# ============================================================

inspection_model = TopKSAE(
    input_shape=384,
    nb_concepts=768,
    top_k=16,
    device=DEVICE
)

print("=" * 70)
print("METAMER-SAE — TOPK SAE ARCHITECTURE AUDIT")
print("=" * 70)

# ============================================================
# 1. CLASS INHERITANCE
# ============================================================

print("\n1. CLASS INHERITANCE")
print("-" * 70)

for cls in type(inspection_model).mro():
    print(cls.__module__, ":", cls.__name__)

# ============================================================
# 2. MODEL MODULE STRUCTURE
# ============================================================

print("\n2. MODEL MODULE STRUCTURE")
print("-" * 70)

print(inspection_model)

# ============================================================
# 3. PARAMETER NAMES AND DIMENSIONS
# ============================================================

print("\n3. MODEL PARAMETERS")
print("-" * 70)

for name, parameter in inspection_model.named_parameters():

    print(
        f"{name:45s} "
        f"{str(tuple(parameter.shape)):20s} "
        f"trainable={parameter.requires_grad}"
    )

# ============================================================
# 4. MODEL BUFFERS
# ============================================================

print("\n4. MODEL BUFFERS")
print("-" * 70)

for name, buffer in inspection_model.named_buffers():
    print(name, tuple(buffer.shape), buffer.dtype)

# ============================================================
# 5. DICTIONARY DIMENSIONS
# ============================================================

print("\n5. DICTIONARY")
print("-" * 70)

dictionary = inspection_model.get_dictionary()

print("Dictionary shape:", tuple(dictionary.shape))
print("Dictionary dtype:", dictionary.dtype)
print("Dictionary device:", dictionary.device)
print("Dictionary finite:", bool(torch.isfinite(dictionary).all()))

# ============================================================
# 6. INHERITED FORWARD AND DECODE IMPLEMENTATIONS
# ============================================================

print("\n6. FORWARD IMPLEMENTATION")
print("-" * 70)

try:
    print(inspect.getsource(type(inspection_model).forward))
except Exception as e:
    print("Could not inspect forward:", e)

print("\n7. DECODE IMPLEMENTATION")
print("-" * 70)

try:
    print(inspect.getsource(type(inspection_model).decode))
except Exception as e:
    print("Could not inspect decode:", e)

# ============================================================
# 8. ENCODER IMPLEMENTATION
# ============================================================

print("\n8. ENCODER IMPLEMENTATION")
print("-" * 70)

try:
    print(inspect.getsource(
        type(inspection_model).encode
    ))
except Exception as e:
    print("Could not inspect encode:", e)

print("\n" + "=" * 70)
print("ARCHITECTURE AUDIT COMPLETE")
print("=" * 70)

del inspection_model, dictionary

METAMER-SAE — TOPK SAE ARCHITECTURE AUDIT

1. CLASS INHERITANCE
----------------------------------------------------------------------
overcomplete.sae.topk_sae : TopKSAE
overcomplete.sae.base : SAE
overcomplete.base : BaseDictionaryLearning
abc : ABC
torch.nn.modules.module : Module
builtins : object

2. MODEL MODULE STRUCTURE
----------------------------------------------------------------------
TopKSAE(
  (encoder): MLPEncoder(
    (mlp_blocks): ModuleList()
    (final_block): Sequential(
      (0): Linear(in_features=384, out_features=768, bias=True)
      (1): Identity()
    )
    (final_activation): ReLU()
  )
  (dictionary): DictionaryLayer()
)

3. MODEL PARAMETERS
----------------------------------------------------------------------
encoder.final_block.0.weight                  (768, 384)           trainable=True
encoder.final_block.0.bias                    (768,)               trainable=True
dictionary._weights                           (768, 384)           trainable=True

4

In [ ]:
import numpy as np
import pandas as pd
import hashlib
import json

# ============================================================
# METAMER-SAE — REAL ACTIVATION DATA PREPARATION
# ============================================================

TRAIN_ACTIVATION_PATH = TRAIN_DIR / "pooled_activations.npy"
VALIDATION_ACTIVATION_PATH = (
    VALIDATION_DIR / "pooled_activations.npy"
)

TRAIN_METADATA_PATH = TRAIN_DIR / "metadata.csv"
VALIDATION_METADATA_PATH = VALIDATION_DIR / "metadata.csv"

# ============================================================
# 1. LOAD ACTIVATIONS USING MEMORY MAPPING
# ============================================================

print("=" * 70)
print("METAMER-SAE — ACTIVATION DATA AUDIT")
print("=" * 70)

X_train = np.load(
    TRAIN_ACTIVATION_PATH,
    mmap_mode="r"
)

X_validation = np.load(
    VALIDATION_ACTIVATION_PATH,
    mmap_mode="r"
)

# ============================================================
# 2. LOAD METADATA
# ============================================================

train_metadata = pd.read_csv(
    TRAIN_METADATA_PATH
)

validation_metadata = pd.read_csv(
    VALIDATION_METADATA_PATH
)

# ============================================================
# 3. EXPECTED SHAPES AND DTYPES
# ============================================================

EXPECTED_TRAIN_SHAPE = (9469, 384)
EXPECTED_VALIDATION_SHAPE = (3925, 384)

assert X_train.shape == EXPECTED_TRAIN_SHAPE, (
    f"Unexpected training shape: {X_train.shape}"
)

assert X_validation.shape == EXPECTED_VALIDATION_SHAPE, (
    f"Unexpected validation shape: {X_validation.shape}"
)

assert X_train.dtype == np.float32
assert X_validation.dtype == np.float32

assert len(train_metadata) == EXPECTED_TRAIN_SHAPE[0]
assert len(validation_metadata) == EXPECTED_VALIDATION_SHAPE[0]

# ============================================================
# 4. FINITE VALUE VALIDATION
# ============================================================

print("\nFINITE VALUE AUDIT")
print("-" * 70)

train_finite = bool(np.isfinite(X_train).all())
validation_finite = bool(np.isfinite(X_validation).all())

print("Training finite:", train_finite)
print("Validation finite:", validation_finite)

assert train_finite
assert validation_finite

# ============================================================
# 5. VERIFY FILE HASHES AGAINST PCA CONFIGURATION
# ============================================================

with open(PCA_RUN_DIR / "pca_run_config.json", "r") as f:
    pca_config = json.load(f)


def calculate_sha256(path):

    sha = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            sha.update(chunk)

    return sha.hexdigest()


files_to_verify = [
    (
        "Training activations",
        TRAIN_ACTIVATION_PATH,
        "fit_activation_sha256"
    ),
    (
        "Validation activations",
        VALIDATION_ACTIVATION_PATH,
        "validation_activation_sha256"
    ),
    (
        "Training metadata",
        TRAIN_METADATA_PATH,
        "train_metadata_sha256"
    ),
    (
        "Validation metadata",
        VALIDATION_METADATA_PATH,
        "validation_metadata_sha256"
    )
]

print("\nSHA-256 INTEGRITY AUDIT")
print("-" * 70)

for label, path, config_key in files_to_verify:

    actual_hash = calculate_sha256(path)
    expected_hash = pca_config[config_key]

    passed = actual_hash == expected_hash

    print(f"\n{label}")
    print("Expected:", expected_hash)
    print("Actual:  ", actual_hash)
    print("Status:  ", "PASS" if passed else "FAIL")

    assert passed, f"SHA-256 mismatch: {label}"

# ============================================================
# 6. CHECK TRAINING AND VALIDATION IDENTITY SEPARATION
# ============================================================

print("\nMETADATA INSPECTION")
print("-" * 70)

print("Training columns:", train_metadata.columns.tolist())
print("Validation columns:", validation_metadata.columns.tolist())

print("\nTraining preview:")
display(train_metadata.head())

print("\nValidation preview:")
display(validation_metadata.head())

# Check hash-based identity overlap if SHA-256 fields exist.

hash_columns = [
    col for col in train_metadata.columns
    if "sha256" in col.lower()
    and col in validation_metadata.columns
]

for column in hash_columns:

    train_ids = set(train_metadata[column].dropna())
    validation_ids = set(validation_metadata[column].dropna())

    overlap = train_ids.intersection(validation_ids)

    print(
        f"\nOverlap in {column}: {len(overlap)}"
    )

    assert len(overlap) == 0, (
        f"Training/validation leakage detected in {column}"
    )

# ============================================================
# 7. ACTIVATION DISTRIBUTION DIAGNOSTICS
# ============================================================

train_mean = np.mean(
    X_train,
    axis=0,
    dtype=np.float64
)

train_std = np.std(
    X_train,
    axis=0,
    dtype=np.float64
)

validation_mean = np.mean(
    X_validation,
    axis=0,
    dtype=np.float64
)

print("\nACTIVATION STATISTICS")
print("-" * 70)

print("Training mean, overall:", float(X_train.mean()))
print("Training standard deviation, overall:", float(X_train.std()))

print("Median channel standard deviation:", float(np.median(train_std)))
print("Maximum channel standard deviation:", float(np.max(train_std)))
print("Minimum channel standard deviation:", float(np.min(train_std)))

print(
    "Training/validation mean absolute difference:",
    float(np.mean(np.abs(train_mean - validation_mean)))
)

# ============================================================
# 8. PCA COMPATIBILITY CHECK
# ============================================================

assert pca_config["input_dimensions"] == X_train.shape[1]

assert pca_config["fit_sample_count"] == len(X_train)

assert pca_config["validation_sample_count"] == len(X_validation)

print("\nPCA COMPATIBILITY")
print("-" * 70)

print("PCA feature dimension:", pca_config["input_dimensions"])
print("SAE input dimension:", X_train.shape[1])
print("Dimensions match:", True)

# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("ACTIVATION DATA AUDIT SUMMARY")
print("=" * 70)

print("Training shape:", X_train.shape)
print("Validation shape:", X_validation.shape)

print("Training dtype:", X_train.dtype)
print("Validation dtype:", X_validation.dtype)

print("Training metadata rows:", len(train_metadata))
print("Validation metadata rows:", len(validation_metadata))

print("All integrity checks passed.")

print("\nFeather primary evaluation loaded: NO")
print("Feather reserve loaded: NO")

print("\n" + "=" * 70)
print("ACTIVATION DATA AUDIT COMPLETE")
print("=" * 70)

METAMER-SAE — ACTIVATION DATA AUDIT

FINITE VALUE AUDIT
----------------------------------------------------------------------
Training finite: True
Validation finite: True

SHA-256 INTEGRITY AUDIT
----------------------------------------------------------------------

Training activations
Expected: eb41bf0e1460401780f617e176af8371c458547b94692df527db9c0376878de0
Actual:   eb41bf0e1460401780f617e176af8371c458547b94692df527db9c0376878de0
Status:   PASS

Validation activations
Expected: ad609ef1e03b6855c8026972844c2e745ba96bae5b06066ee838469d880c2260
Actual:   ad609ef1e03b6855c8026972844c2e745ba96bae5b06066ee838469d880c2260
Status:   PASS

Training metadata
Expected: 4006103141da0915df33d12425ce2679114ad154d81df8dc124cb61c06a45864
Actual:   4006103141da0915df33d12425ce2679114ad154d81df8dc124cb61c06a45864
Status:   PASS

Validation metadata
Expected: 17128d4869d8d18389655068de19f74a22dfc5beb16e1e04b300610788882553
Actual:   17128d4869d8d18389655068de19f74a22dfc5beb16e1e04b300610788882553


,sample_index,metadata_json
0,0,"{""split"": ""train"", ""class"": ""n01440764"", ""file..."
1,1,"{""split"": ""train"", ""class"": ""n01440764"", ""file..."
2,2,"{""split"": ""train"", ""class"": ""n01440764"", ""file..."
3,3,"{""split"": ""train"", ""class"": ""n01440764"", ""file..."
4,4,"{""split"": ""train"", ""class"": ""n01440764"", ""file..."



Validation preview:


,sample_index,metadata_json
0,0,"{""split"": ""val"", ""class"": ""n01440764"", ""filena..."
1,1,"{""split"": ""val"", ""class"": ""n01440764"", ""filena..."
2,2,"{""split"": ""val"", ""class"": ""n01440764"", ""filena..."
3,3,"{""split"": ""val"", ""class"": ""n01440764"", ""filena..."
4,4,"{""split"": ""val"", ""class"": ""n01440764"", ""filena..."



ACTIVATION STATISTICS
----------------------------------------------------------------------
Training mean, overall: 0.08893764764070511
Training standard deviation, overall: 1.8687303066253662
Median channel standard deviation: 0.07588691157298684
Maximum channel standard deviation: 5.767212920413322
Minimum channel standard deviation: 0.03677829251178864
Training/validation mean absolute difference: 0.0012795419087243166

PCA COMPATIBILITY
----------------------------------------------------------------------
PCA feature dimension: 384
SAE input dimension: 384
Dimensions match: True

ACTIVATION DATA AUDIT SUMMARY
Training shape: (9469, 384)
Validation shape: (3925, 384)
Training dtype: float32
Validation dtype: float32
Training metadata rows: 9469
Validation metadata rows: 3925
All integrity checks passed.

Feather primary evaluation loaded: NO
Feather reserve loaded: NO

ACTIVATION DATA AUDIT COMPLETE


### Cell 9 — Inspect dictionary normalisation and multiplier

In [ ]:
import inspect
import torch
from torch import nn
from overcomplete.sae import TopKSAE

print("=" * 70)
print("METAMER-SAE — DICTIONARY IMPLEMENTATION AUDIT")
print("=" * 70)

audit_model = TopKSAE(
    input_shape=384,
    nb_concepts=768,
    top_k=16,
    device=DEVICE
)

dictionary_layer = audit_model.dictionary

print("\n1. DICTIONARY CONFIGURATION")
print("-" * 70)

print("Dictionary class:", type(dictionary_layer).__name__)

print(
    "Normalisation function:",
    getattr(
        dictionary_layer.normalization,
        "__name__",
        str(dictionary_layer.normalization)
    )
)

print(
    "Multiplier:",
    dictionary_layer.multiplier.detach().cpu().item()
)

print(
    "Multiplier trainable:",
    isinstance(dictionary_layer.multiplier, nn.Parameter)
    and dictionary_layer.multiplier.requires_grad
)

print("\n2. DICTIONARY IMPLEMENTATION")
print("-" * 70)

print(inspect.getsource(type(dictionary_layer)))

print("\n3. NORMALISED DICTIONARY")
print("-" * 70)

audit_model.train()

dictionary = audit_model.get_dictionary()

atom_norms = torch.linalg.vector_norm(
    dictionary,
    ord=2,
    dim=1
)

print("Dictionary shape:", tuple(dictionary.shape))
print("Minimum atom norm:", atom_norms.min().item())
print("Median atom norm:", atom_norms.median().item())
print("Maximum atom norm:", atom_norms.max().item())

print("\n4. DECODER BIAS")
print("-" * 70)

print(
    "Separate decoder bias:",
    any(
        "bias" in name.lower()
        for name, _ in audit_model.dictionary.named_parameters()
    )
)

print("\n" + "=" * 70)
print("DICTIONARY AUDIT COMPLETE")
print("=" * 70)

del audit_model, dictionary_layer, dictionary, atom_norms

METAMER-SAE — DICTIONARY IMPLEMENTATION AUDIT

1. DICTIONARY CONFIGURATION
----------------------------------------------------------------------
Dictionary class: DictionaryLayer
Normalisation function: normalize_l2
Multiplier: 0.0
Multiplier trainable: False

2. DICTIONARY IMPLEMENTATION
----------------------------------------------------------------------
class DictionaryLayer(nn.Module):
    """
    A neural network layer representing a dictionary for reconstructing input data.

    Parameters
    ----------
    in_dimensions : int
        Dimensionality of the input data (e.g number of channels).
    nb_concepts : int
        Number of components/concepts in the dictionary. The dictionary is overcomplete if
        the number of concepts > in_dimensions.
    normalization : str or callable, optional
        Whether to normalize the dictionary, by default 'l2' normalization is applied.
        Current options are 'l2', 'max_l2', 'l1', 'max_l1', 'identity'.
        If a custom norm

### Cell 10 — Correct nested metadata and split audit

In [ ]:
import json
import pandas as pd

print("=" * 70)
print("METAMER-SAE — NESTED METADATA AUDIT")
print("=" * 70)

# Parse the JSON metadata into ordinary dataframe columns.

def parse_metadata(df):

    records = df["metadata_json"].apply(
        lambda x: json.loads(x)
        if isinstance(x, str)
        else x
    )

    return pd.json_normalize(records)


train_details = parse_metadata(train_metadata)
validation_details = parse_metadata(validation_metadata)

print("\nAVAILABLE METADATA")
print("-" * 70)

print("Training:", train_details.columns.tolist())
print("Validation:", validation_details.columns.tolist())

# Verify split labels.

assert set(train_details["split"].dropna().unique()) == {"train"}

assert set(validation_details["split"].dropna().unique()) == {"val"}

print("\nSPLIT VERIFICATION")
print("-" * 70)

print("Training split:", train_details["split"].unique())
print("Validation split:", validation_details["split"].unique())

# Search for available image-identification fields.

identity_candidates = [
    column for column in train_details.columns
    if column in validation_details.columns
    and any(
        term in column.lower()
        for term in [
            "filename",
            "file_name",
            "filepath",
            "relative_path",
            "sha256",
            "image_id",
            "sample_id"
        ]
    )
]

print("\nIDENTITY SEPARATION")
print("-" * 70)

if not identity_candidates:

    print(
        "No shared image-identity fields were found. "
        "Identity overlap remains unverified."
    )

else:

    for column in identity_candidates:

        train_values = set(
            train_details[column].dropna().astype(str)
        )

        validation_values = set(
            validation_details[column].dropna().astype(str)
        )

        overlap = train_values.intersection(validation_values)

        print(f"\nField: {column}")
        print("Training unique values:", len(train_values))
        print("Validation unique values:", len(validation_values))
        print("Overlap:", len(overlap))

        assert len(overlap) == 0, (
            f"Identity overlap found in {column}"
        )

# Inspect class allocation.

print("\nCLASS DISTRIBUTION")
print("-" * 70)

print("\nTraining:")
display(train_details["class"].value_counts().sort_index())

print("\nValidation:")
display(validation_details["class"].value_counts().sort_index())

print("\n" + "=" * 70)
print("NESTED METADATA AUDIT COMPLETE")
print("=" * 70)

METAMER-SAE — NESTED METADATA AUDIT

AVAILABLE METADATA
----------------------------------------------------------------------
Training: ['split', 'class', 'filename', 'relative_path']
Validation: ['split', 'class', 'filename', 'relative_path']

SPLIT VERIFICATION
----------------------------------------------------------------------
Training split: ['train']
Validation split: ['val']

IDENTITY SEPARATION
----------------------------------------------------------------------

Field: filename
Training unique values: 9469
Validation unique values: 3925
Overlap: 0

Field: relative_path
Training unique values: 9469
Validation unique values: 3925
Overlap: 0

CLASS DISTRIBUTION
----------------------------------------------------------------------

Training:


,count
class,
n01440764,963
n02102040,955
n02979186,993
n03000684,858
n03028079,941
n03394916,956
n03417042,961
n03425413,931
n03445777,951



Validation:


,count
class,
n01440764,387
n02102040,395
n02979186,357
n03000684,386
n03028079,409
n03394916,394
n03417042,389
n03425413,419
n03445777,399



NESTED METADATA AUDIT COMPLETE


### 5. Cell 11 — Apply mean-centering and save preprocessing artifacts

In [ ]:
import numpy as np
import json
import hashlib
from datetime import datetime, timezone

# ============================================================
# METAMER-SAE — SAE PREPROCESSING
# ============================================================

print("=" * 70)
print("METAMER-SAE — TRAINING-ONLY MEAN CENTERING")
print("=" * 70)

# Calculate mean using training data only, with float64
# accumulation for numerical precision.

TRAIN_MEAN = np.mean(
    X_train,
    axis=0,
    dtype=np.float64
).astype(np.float32)

# Apply the same training mean to both datasets.

X_train_centered = np.asarray(
    X_train,
    dtype=np.float32
) - TRAIN_MEAN[None, :]

X_validation_centered = np.asarray(
    X_validation,
    dtype=np.float32
) - TRAIN_MEAN[None, :]

# ============================================================
# VALIDATION
# ============================================================

assert X_train_centered.shape == (9469, 384)
assert X_validation_centered.shape == (3925, 384)

assert np.isfinite(X_train_centered).all()
assert np.isfinite(X_validation_centered).all()

centered_train_mean = np.mean(
    X_train_centered,
    axis=0,
    dtype=np.float64
)

max_residual_mean = float(
    np.max(np.abs(centered_train_mean))
)

assert max_residual_mean < 1e-5, (
    "Training centering verification failed."
)

print("\nPREPROCESSING RESULTS")
print("-" * 70)

print("Training shape:", X_train_centered.shape)
print("Validation shape:", X_validation_centered.shape)

print("Training residual mean:", float(
    np.mean(X_train_centered)
))

print("Maximum absolute channel mean:", max_residual_mean)

print("Training centred std:", float(X_train_centered.std()))
print("Validation centred std:", float(X_validation_centered.std()))

# ============================================================
# SAVE PREPROCESSING ARTIFACTS
# ============================================================

MEAN_PATH = (
    SAE_PREPARATION_DIR / "training_activation_mean.npy"
)

TRAIN_CENTERED_PATH = (
    SAE_PREPARATION_DIR / "train_centered_activations.npy"
)

VALIDATION_CENTERED_PATH = (
    SAE_PREPARATION_DIR / "validation_centered_activations.npy"
)

np.save(MEAN_PATH, TRAIN_MEAN)
np.save(TRAIN_CENTERED_PATH, X_train_centered)
np.save(VALIDATION_CENTERED_PATH, X_validation_centered)

# ============================================================
# HASH HELPER
# ============================================================

def sha256_file(path):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            sha.update(chunk)

    return sha.hexdigest()

# ============================================================
# SAVE PREPROCESSING CONFIGURATION
# ============================================================

preprocessing_config = {
    "project": "Metamer-SAE",
    "notebook": "05_overcomplete_environment_and_sae_preparation.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "preprocessing_method": "training_mean_centering_only",

    "input_dimensions": 384,

    "training_samples": 9469,
    "validation_samples": 3925,

    "mean_fit_dataset": "Imagenette-320 training",
    "mean_fit_samples": 9469,

    "per_channel_standardization": False,
    "whitening": False,

    "training_mean_path": str(MEAN_PATH),
    "training_mean_sha256": sha256_file(MEAN_PATH),

    "centered_training_path": str(TRAIN_CENTERED_PATH),
    "centered_training_sha256": sha256_file(TRAIN_CENTERED_PATH),

    "centered_validation_path": str(VALIDATION_CENTERED_PATH),
    "centered_validation_sha256": sha256_file(
        VALIDATION_CENTERED_PATH
    ),

    "original_training_sha256": pca_config[
        "fit_activation_sha256"
    ],

    "original_validation_sha256": pca_config[
        "validation_activation_sha256"
    ],

    "future_feather_transform": (
        "Subtract the saved training_activation_mean.npy"
    )
}

PREPROCESSING_CONFIG_PATH = (
    SAE_PREPARATION_DIR / "sae_preprocessing_config.json"
)

with open(PREPROCESSING_CONFIG_PATH, "w") as f:
    json.dump(preprocessing_config, f, indent=4)

print("\nSAVED ARTIFACTS")
print("-" * 70)

for path in [
    MEAN_PATH,
    TRAIN_CENTERED_PATH,
    VALIDATION_CENTERED_PATH,
    PREPROCESSING_CONFIG_PATH
]:

    print(path.name, ":", path.stat().st_size / (1024**2), "MB")

print("\n" + "=" * 70)
print("SAE PREPROCESSING COMPLETE")
print("=" * 70)

METAMER-SAE — TRAINING-ONLY MEAN CENTERING

PREPROCESSING RESULTS
----------------------------------------------------------------------
Training shape: (9469, 384)
Validation shape: (3925, 384)
Training residual mean: 7.63758611910248e-10
Maximum absolute channel mean: 3.6257551368280706e-07
Training centred std: 0.3058968782424927
Validation centred std: 0.30161380767822266

SAVED ARTIFACTS
----------------------------------------------------------------------
training_activation_mean.npy : 0.0015869140625 MB
train_centered_activations.npy : 13.8707275390625 MB
validation_centered_activations.npy : 5.7496337890625 MB
sae_preprocessing_config.json : 0.0013456344604492188 MB

SAE PREPROCESSING COMPLETE


### 6. Cell 12 — Create SAE DataLoaders
Our activation dataset is relatively small, so we can keep the centred matrices in CPU memory and transfer batches to the GPU during training.

I recommend:
- Batch size: 256.
- Training shuffle: enabled.
- Validation shuffle: disabled.
- Workers: 0.
- Pinned memory: enabled when CUDA is available.

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# ============================================================
# METAMER-SAE — DATA LOADER PREPARATION
# ============================================================

print("=" * 70)
print("METAMER-SAE — SAE DATALOADER PREPARATION")
print("=" * 70)

BATCH_SIZE = 256
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# Convert centred NumPy arrays into CPU tensors.

train_tensor = torch.from_numpy(
    np.ascontiguousarray(X_train_centered)
)

validation_tensor = torch.from_numpy(
    np.ascontiguousarray(X_validation_centered)
)

# Wrap datasets.

train_dataset = TensorDataset(train_tensor)

validation_dataset = TensorDataset(validation_tensor)

# Reproducible DataLoader shuffle.

loader_generator = torch.Generator()
loader_generator.manual_seed(42)

# Training DataLoader.

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
    generator=loader_generator
)

# Validation DataLoader.

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False
)

# ============================================================
# FIRST-BATCH VALIDATION
# ============================================================

train_batch = next(iter(train_loader))[0]

validation_batch = next(iter(validation_loader))[0]

assert train_batch.shape == (256, 384)
assert validation_batch.shape == (256, 384)

assert torch.isfinite(train_batch).all()
assert torch.isfinite(validation_batch).all()

# ============================================================
# SUMMARY
# ============================================================

print("\nDATALOADER CONFIGURATION")
print("-" * 70)

print("Device:", DEVICE)
print("Batch size:", BATCH_SIZE)
print("Workers:", NUM_WORKERS)
print("Pinned memory:", PIN_MEMORY)

print("\nTRAINING")
print("-" * 70)

print("Samples:", len(train_dataset))
print("Batches per epoch:", len(train_loader))
print("First batch shape:", tuple(train_batch.shape))

print("\nVALIDATION")
print("-" * 70)

print("Samples:", len(validation_dataset))
print("Batches per epoch:", len(validation_loader))
print("First batch shape:", tuple(validation_batch.shape))

print("\n" + "=" * 70)
print("DATALOADER PREPARATION COMPLETE")
print("=" * 70)

METAMER-SAE — SAE DATALOADER PREPARATION

DATALOADER CONFIGURATION
----------------------------------------------------------------------
Device: cuda
Batch size: 256
Workers: 0
Pinned memory: True

TRAINING
----------------------------------------------------------------------
Samples: 9469
Batches per epoch: 37
First batch shape: (256, 384)

VALIDATION
----------------------------------------------------------------------
Samples: 3925
Batches per epoch: 16
First batch shape: (256, 384)

DATALOADER PREPARATION COMPLETE


### 7. Cell 13 — Establish the initial SAE experiment configurations
Now we need to define what we intend to train in Notebook 06.

The handover recommends two overcompleteness factors:
- 2×: 768 latent features.
- 4×: 1,536 latent features.

In [ ]:
import json

# ============================================================
# METAMER-SAE — SAE EXPERIMENT CONFIGURATION
# ============================================================

print("=" * 70)
print("METAMER-SAE — TOPK SAE EXPERIMENT CONFIGURATION")
print("=" * 70)

INPUT_DIM = 384

# ============================================================
# DICTIONARY CONFIGURATIONS
# ============================================================

DICTIONARY_CONFIGS = [
    {
        "expansion_factor": 2,
        "nb_concepts": 768,
        "top_k_values": [16, 32]
    },
    {
        "expansion_factor": 4,
        "nb_concepts": 1536,
        "top_k_values": [32, 64]
    }
]

# ============================================================
# TRAINING CONFIGURATION
# ============================================================

SAE_TRAINING_CONFIG = {

    "input_dimensions": INPUT_DIM,

    "dictionary_normalization": "l2",

    "use_multiplier": False,

    "encoder": "default_overcomplete_mlp",

    "activation": "ReLU",

    "optimizer": "Adam",

    "learning_rate": 0.001,

    "weight_decay": 0.0,

    "batch_size": BATCH_SIZE,

    "max_epochs": 100,

    "gradient_clipping": 1.0,

    "validation_frequency": 1,

    "early_stopping_patience": 10,

    "loss": "reconstruction_MSE",

    "random_seeds": [42, 123],

    "device": DEVICE
}

# ============================================================
# CREATE EXPERIMENT GRID
# ============================================================

EXPERIMENT_GRID = []

for dictionary_config in DICTIONARY_CONFIGS:

    for top_k in dictionary_config["top_k_values"]:

        active_fraction = (
            top_k / dictionary_config["nb_concepts"]
        )

        for seed in SAE_TRAINING_CONFIG["random_seeds"]:

            experiment = {

                "input_dimensions": INPUT_DIM,

                "expansion_factor": (
                    dictionary_config["expansion_factor"]
                ),

                "nb_concepts": (
                    dictionary_config["nb_concepts"]
                ),

                "top_k": top_k,

                "active_fraction": active_fraction,

                "random_seed": seed
            }

            EXPERIMENT_GRID.append(experiment)

# ============================================================
# DISPLAY EXPERIMENT GRID
# ============================================================

print("\nINITIAL TRAINING EXPERIMENTS")
print("-" * 70)

for index, experiment in enumerate(EXPERIMENT_GRID, start=1):

    print(
        f"Experiment {index}: "
        f"{experiment['expansion_factor']}x | "
        f"Dictionary={experiment['nb_concepts']} | "
        f"TopK={experiment['top_k']} | "
        f"Active={experiment['active_fraction']:.2%} | "
        f"Seed={experiment['random_seed']}"
    )

print("\nTotal experiments:", len(EXPERIMENT_GRID))

assert len(EXPERIMENT_GRID) == 8

# ============================================================
# SAVE CONFIGURATION
# ============================================================

experiment_config_path = (
    SAE_PREPARATION_DIR / "topk_sae_experiment_config.json"
)

experiment_record = {
    "project": "Metamer-SAE",
    "status": "proposed_initial_training_grid",
    "training_configuration": SAE_TRAINING_CONFIG,
    "dictionary_configurations": DICTIONARY_CONFIGS,
    "experiment_grid": EXPERIMENT_GRID
}

with open(experiment_config_path, "w") as f:

    json.dump(
        experiment_record,
        f,
        indent=4
    )

print("\nSaved:", experiment_config_path)

print("\n" + "=" * 70)
print("EXPERIMENT CONFIGURATION SAVED")
print("=" * 70)

METAMER-SAE — TOPK SAE EXPERIMENT CONFIGURATION

INITIAL TRAINING EXPERIMENTS
----------------------------------------------------------------------
Experiment 1: 2x | Dictionary=768 | TopK=16 | Active=2.08% | Seed=42
Experiment 2: 2x | Dictionary=768 | TopK=16 | Active=2.08% | Seed=123
Experiment 3: 2x | Dictionary=768 | TopK=32 | Active=4.17% | Seed=42
Experiment 4: 2x | Dictionary=768 | TopK=32 | Active=4.17% | Seed=123
Experiment 5: 4x | Dictionary=1536 | TopK=32 | Active=2.08% | Seed=42
Experiment 6: 4x | Dictionary=1536 | TopK=32 | Active=2.08% | Seed=123
Experiment 7: 4x | Dictionary=1536 | TopK=64 | Active=4.17% | Seed=42
Experiment 8: 4x | Dictionary=1536 | TopK=64 | Active=4.17% | Seed=123

Total experiments: 8

Saved: /content/drive/MyDrive/metamer-sae/outputs/sae_preparation/topk_sae_experiment_config.json

EXPERIMENT CONFIGURATION SAVED


### 8. Cell 14 — Notebook 05 final preparation audit
Finally, we need to verify that the complete preparation pipeline has produced its expected persistent artifacts.

In [ ]:
import json
import torch
import numpy as np

# ============================================================
# METAMER-SAE — NOTEBOOK 05 CLOSURE AUDIT
# ============================================================

print("=" * 70)
print("METAMER-SAE — NOTEBOOK 05 FINAL VERIFICATION")
print("=" * 70)

required_outputs = [
    MEAN_PATH,
    TRAIN_CENTERED_PATH,
    VALIDATION_CENTERED_PATH,
    PREPROCESSING_CONFIG_PATH,
    experiment_config_path
]

print("\n1. ARTIFACT VERIFICATION")
print("-" * 70)

for path in required_outputs:

    assert path.exists(), f"Missing artifact: {path}"

    print(
        f"{path.name:45s} "
        f"{path.stat().st_size / (1024**2):.2f} MB"
    )

# ============================================================
# 2. PREPROCESSING VERIFICATION
# ============================================================

saved_mean = np.load(MEAN_PATH)

saved_train = np.load(TRAIN_CENTERED_PATH, mmap_mode="r")

saved_validation = np.load(
    VALIDATION_CENTERED_PATH,
    mmap_mode="r"
)

assert saved_mean.shape == (384,)
assert saved_train.shape == (9469, 384)
assert saved_validation.shape == (3925, 384)

assert np.isfinite(saved_mean).all()
assert np.isfinite(saved_train).all()
assert np.isfinite(saved_validation).all()

# ============================================================
# 3. DATALOADER VERIFICATION
# ============================================================

assert len(train_loader) == 37
assert len(validation_loader) == 16

# ============================================================
# 4. EXPERIMENT GRID VERIFICATION
# ============================================================

assert len(EXPERIMENT_GRID) == 8

assert len(SAE_TRAINING_CONFIG["random_seeds"]) == 2

# ============================================================
# 5. PRIMARY/RESERVE SEPARATION
# ============================================================

print("\nDATA SEPARATION")
print("-" * 70)

print("Imagenette training: Used for preprocessing")
print("Imagenette validation: Prepared for model selection")
print("Feather development: Not loaded")
print("Feather primary evaluation: Not loaded")
print("Feather reserve: Not loaded")

# ============================================================
# 6. SAVE CLOSURE REPORT
# ============================================================

closure_report = {

    "project": "Metamer-SAE",

    "notebook": (
        "05_overcomplete_environment_and_sae_preparation.ipynb"
    ),

    "environment_compatible": True,

    "overcomplete_version": "0.3.0",

    "topk_api_smoke_test": "passed",

    "dictionary_audit": "completed",

    "activation_integrity": "passed",

    "metadata_split_audit": "passed",

    "preprocessing": "training_mean_centering_only",

    "train_samples": len(train_dataset),

    "validation_samples": len(validation_dataset),

    "input_dimensions": INPUT_DIM,

    "train_batches": len(train_loader),

    "validation_batches": len(validation_loader),

    "initial_experiment_runs": len(EXPERIMENT_GRID),

    "primary_evaluation_used": False,

    "reserve_used": False,

    "full_sae_training_performed": False
}

closure_path = (
    SAE_PREPARATION_DIR / "notebook_05_closure_report.json"
)

with open(closure_path, "w") as f:

    json.dump(
        closure_report,
        f,
        indent=4
    )

print("\n" + "=" * 70)
print("NOTEBOOK 05 PREPARATION AUDIT PASSED")
print("=" * 70)

print("Closure report:", closure_path)

METAMER-SAE — NOTEBOOK 05 FINAL VERIFICATION

1. ARTIFACT VERIFICATION
----------------------------------------------------------------------
training_activation_mean.npy                  0.00 MB
train_centered_activations.npy                13.87 MB
validation_centered_activations.npy           5.75 MB
sae_preprocessing_config.json                 0.00 MB
topk_sae_experiment_config.json               0.00 MB

DATA SEPARATION
----------------------------------------------------------------------
Imagenette training: Used for preprocessing
Imagenette validation: Prepared for model selection
Feather development: Not loaded
Feather primary evaluation: Not loaded
Feather reserve: Not loaded

NOTEBOOK 05 PREPARATION AUDIT PASSED
Closure report: /content/drive/MyDrive/metamer-sae/outputs/sae_preparation/notebook_05_closure_report.json
